In [1]:
#参考https://github.com/bbruceyuan/LLMs-Zero-to-Hero/blob/master/src/video/build_gpt.ipynb
#参考Dong, L., Yang, N., Wang, W., Wei, F., Liu, X., Wang, Y., Gao, J., Zhou, M. and Hon, H.W., 2019. 
#Unified language model pre-training for natural language understanding and generation. 
#Advances in neural information processing systems, 32.中的Unifi模型

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset
from torch.utils.data import DataLoader
from dataclasses import dataclass
import math
torch.manual_seed(1)

In [2]:
@dataclass
class DataConfig:
    block_size: int = 32

In [3]:
class SingleHeadAttention(nn.Module):
    # 单头注意力
    def __init__(self, config):
        super().__init__()
        self.key = nn.Linear(config.n_embd, config.head_size)
        self.value = nn.Linear(config.n_embd, config.head_size)
        self.query = nn.Linear(config.n_embd, config.head_size)
        self.head_size = config.head_size

        
        # need to understand!!!
        self.register_buffer(
            'attention_mask',
            torch.tril(
                torch.ones(config.block_size, config.block_size)
            ))
        # 设置掩码的一串代码
        self.dropout = nn.Dropout(config.dropout)

    def forward(self, x):
        batch_size, seq_len, hidden_size = x.size()
        # 运用Module模块自带的线性层设置参数与进行矩阵乘法
        k = self.key(x)
        v = self.value(x)
        q = self.query(x)

        weight = q @ k.transpose(-2, -1)

        weight = weight.masked_fill(
            self.attention_mask[:seq_len, :seq_len] == 0,
            float('-inf')
        ) / math.sqrt(self.head_size) # 这里的 hidden_size 其实是 head_size，因为是单头
        weight = F.softmax(weight, dim=-1)
        weight = self.dropout(weight)
        out = weight @ v
        return out 

In [4]:
class MultiHeadAttention(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.heads = nn.ModuleList(
            [
                SingleHeadAttention(config)
                for _ in range(config.n_head)
            ]
        )
        self.proj = nn.Linear(config.n_embd, config.n_embd)
        self.dropout = nn.Dropout(config.dropout)

    def forward(self, x):
        output = torch.cat(
            [h(x) for h in self.heads],
            dim=-1
        )
        output = self.proj(output)
        output = self.dropout(output)
        return output      

In [5]:
class FeedForward(nn.Module):
    # 实际上为MLP
    def __init__(self, config):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(config.n_embd, 4 * config.n_embd),
            nn.GELU(),
            nn.Linear(4 * config.n_embd, config.n_embd),
            nn.Dropout(config.dropout)
        )

    def forward(self, x):
        return self.net(x)

In [6]:
class Block(nn.Module):
    def __init__(self, config):
        super().__init__()
        head_size = config.n_embd // config.n_head
        self.att = MultiHeadAttention(config)
        self.ffn = FeedForward(config)
        self.ln1 = nn.LayerNorm(config.n_embd)
        self.ln2 = nn.LayerNorm(config.n_embd)

    def forward(self, x):
        x = x + self.att(self.ln1(x))
        x = x + self.ffn(self.ln2(x))
        return x

In [7]:
class TokenEmbedding(nn.Module):
    # 为了继承nn.Module中foward函数的用法,更加方便，代码更加简洁
    def __init__(self, vocab_size: int, n_embd: int):
        super().__init__()
        self.vocab_size = vocab_size
        self.n_embd = n_embd
        self.weight = nn.Parameter(torch.Tensor(vocab_size, n_embd))

    
    def forward(self, idx):
        return self.weight[idx]    


class PositionalEmbedding(nn.Module):
    def __init__(self, block_size: int, n_embd:int):
        super().__init__()
        self.block_size = block_size
        self.n_embd = n_embd #无用
        self.pe = torch.zeros(block_size, n_embd) 

        self.pos = torch.arange(0, block_size).unsqueeze(1)
        div_term = torch.exp( torch.arange(0, n_embd, 2) * - (math.log(10000)) / n_embd)
        self.pe[:, 0::2] = torch.sin(self.pos * div_term)
        self.pe[:, 1::2] = torch.cos(self.pos * div_term)
        self.pe = self.pe.to("cuda")

    def forward(self, pos):
        return self.pe[pos]


In [8]:
class MyDataset(Dataset):
    def __init__(self, block_size=512):
        self.block_size = block_size
        self.encoded_data = []
        self.wordstonum = {}
        self.wordstonum[''] = 0
        self.numtowords = {}
        self.numtowords[0]  = ''
        self.total_words = 0

        self.max_lines = 10000
        raw_data = []

        import os
        self.path = r"C:\Users\余全霖\Desktop\人工智能导论\pj\dataset"
        files = os.listdir(self.path)
        
        for file in files:
            file_path = os.path.join(self.path, file)
            with open(file_path, 'r') as f:
                for i, line in enumerate(f):
                    if i >= self.max_lines:
                        break
                    text = line.strip()
                    if text != '':
                        raw_data.append(text)
                        for token in text:
                            if token not in self.wordstonum.keys():
                                self.total_words += 1
                                self.wordstonum[token] = self.total_words
                                self.numtowords[self.total_words] = token
        self.total_words += 1                       

        full_encoded = []
        for text in raw_data:
            encoded_text = self.encode(text)
            full_encoded.extend(encoded_text + [0])

            
        # 将长文本分割成训练样本
        for i in range(0, len(full_encoded), self.block_size):
            #多取一个token作为目标
            chunk = full_encoded[i:i+self.block_size+1]
            # 如果长度不够，用 eos_token 填充，所以所有输入的长度均为block_size
            if len(chunk) < self.block_size + 1:
                chunk = chunk + [0] * (self.block_size + 1 - len(chunk))
            self.encoded_data.append(chunk)

        #这一步已经做到了将不同长度的句子给padding了
    
    def __len__(self):
        return len(self.encoded_data)
    
    def __getitem__(self, idx):
        chunk = self.encoded_data[idx]
        x = torch.tensor(chunk[:-1], dtype=torch.long)
        y = torch.tensor(chunk[1:], dtype=torch.long)
        return x, y

    def encode(self, text):
        """将文本编码为token IDs"""
        ids = []
        for token in text:
            ids.append(self.wordstonum[token])
        
        return ids

    def decode(self, ids):
        """将token IDs解码为文本"""
        text = []
        for num in ids:
            text.append(self.numtowords[num])
        
        return text

In [9]:
class GPT(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.token_embedding_table = TokenEmbedding(config.vocab_size, config.n_embd)
        self.position_embeding_table = PositionalEmbedding(config.block_size, config.n_embd)
        self.blocks = nn.Sequential(
            *[Block(config) for _ in range(config.n_layer)]
        )
        self.ln_final = nn.LayerNorm(config.n_embd)
        self.lm_head = nn.Linear(config.n_embd, config.vocab_size, bias=False)

        self.apply(self._init_weights)
        self.block_size = config.block_size
        self.batch_size = config.batch_size

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)
        elif isinstance(module, TokenEmbedding):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, idx, targets=None):
        # idx 为输入的 token ids
        # print(idx)
        batch, seq_len = idx.size()
        token_emb = self.token_embedding_table(idx)

        pos_emb = self.position_embeding_table(
            torch.arange(seq_len, device=idx.device)
        )

        x = token_emb + pos_emb
        x = self.blocks(x)
        x = self.ln_final(x)
        logits = self.lm_head(x)

        if targets is None:
            loss = None
        else:
            batch, seq_len, vocab_size = logits.size()
            logits = logits.view(batch * seq_len, vocab_size)
            targets = targets.view(batch * seq_len)
            loss = F.cross_entropy(logits, targets)

        return logits, loss

    def generate(self, words_dict, text, max_length):
        lenth = len(text)

        idx = []
        seq = []
        for token in text:
            seq.append(words_dict.wordstonum[token])
        if len(text) < self.block_size:
            seq = [0] * (self.block_size - len(text)) + seq
        for _ in range(self.batch_size):
            idx.append(seq)

        idx = torch.tensor(idx).to("cuda")

        for _ in range(max_length):
            idx_cond = idx if idx.size(1) <= self.block_size else idx[:, -self.block_size:]
            logits, _ = self(idx_cond)
            logits = logits[:, -1, :]  # becomes (B, vocab_size)
            probs = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)  # (B, 1)
            idx = torch.cat((idx, idx_next), dim=1)  # (B, T+1)
        if lenth < self.block_size:
            idx = idx[:, self.block_size - lenth:]

        for i in range(self.batch_size):
            for token in idx[i]:
                print(words_dict.numtowords[int(token)], end = '')
            print("")
            
        return 0
       

In [10]:
train_dataset = MyDataset(DataConfig.block_size)
words_dict = train_dataset
train_dataset, val_dataset = torch.utils.data.random_split(train_dataset, [0.9, 0.1])

train_loader = DataLoader(train_dataset, batch_size=12, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=12, shuffle=False)

In [11]:
@dataclass
class GPTConfig:
    block_size: int = DataConfig.block_size #32
    batch_size: int = 12
    n_layer: int = 6
    n_head: int = 12
    n_embd: int = 768
    head_size: int = n_embd // n_head
    dropout: float = 0.1

    vocab_size: int = words_dict.total_words

In [12]:
model = GPT(GPTConfig())
device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)
print(device)

# 打印模型参数

total_params = sum(p.numel() for p in model.parameters())
print(f"Total parameters: {total_params/1e6} M")
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=1000)

cuda
Total parameters: 46.580736 M


In [13]:
# 训练循环

def train(model, optimizer, scheduler, train_loader, val_loader, device):
    model.train()
    total_loss = 0
    for batch_idx, (x, y) in enumerate(train_loader):
        # 将数据移到设备上
        #print(x)
        #print(y)
        x, y = x.to(device), y.to(device)
        #print(y)
        # 前向传播
        logits, loss = model(x, targets=y)
        # 反向传播
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        # 调整学习率
        scheduler.step()
        
        total_loss += loss.item()
        if batch_idx % 200 == 0:
            model.generate(words_dict, "少年缓缓抬起头来，露出一张有些清秀的稚嫩脸庞，漆黑的眸子木然的在周围那些嘲讽的同龄人身上扫过，", 32)
            print(f'Epoch: {epoch}, Batch: {batch_idx}, Loss: {loss.item():.4f}')
    return total_loss

def eval(model, val_loader, device):
    # 验证
    model.eval()
    val_loss = 0
    with torch.no_grad():
        for x, y in val_loader:
            x, y = x.to(device), y.to(device)
            logits, loss = model(x, targets=y)
            val_loss += loss.item()
    return val_loss

ppl_train = []
ppl_eval = []

for epoch in range(20):
    train_loss = train(model, optimizer, scheduler, train_loader, val_loader, device)
    val_loss = eval(model, val_loader, device)
    ppl_train.append(math.exp(train_loss/len(train_loader)))
    ppl_eval.append(math.exp(val_loss/len(val_loader)))
    print("==============================================")
    model.generate(words_dict, "少年缓缓抬起头来，露出一张有些清秀的稚嫩脸庞，漆黑的眸子木然的在周围那些嘲讽的同龄人身上扫过，", 32)
    print(f'Epoch: {epoch}, Train Loss: {train_loss/len(train_loader):.4f}, Val Loss: {val_loss/len(val_loader):.4f}')
    print("==============================================")

    # 保存模型
    avg_val_loss = val_loss / len(val_loader)
    checkpoint = {
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'scheduler_state_dict': scheduler.state_dict(),
        'val_loss': avg_val_loss,
    }
    # 保存每个epoch的模型
    torch.save(checkpoint, f'C:\\Users\\余全霖\\Desktop\\人工智能导论\\pj\\models\\model_epoch_{epoch}.pt')

少年缓缓抬起头来，露出一张有些清秀的稚嫩脸庞，漆黑的眸子木然的在周围那些嘲讽的同龄人身上扫过，，祝囊代障老述惨，陌，挫，，赢孤郁惬魄怒众瞟搂拍有途疗乾敛曰爽给
少年缓缓抬起头来，露出一张有些清秀的稚嫩脸庞，漆黑的眸子木然的在周围那些嘲讽的同龄人身上扫过，纤恒拉，尊刺庇烧板，卫瞥煌邀米烧大汤挑。窕抬总检渐猪，希惘凭姐可
少年缓缓抬起头来，露出一张有些清秀的稚嫩脸庞，漆黑的眸子木然的在周围那些嘲讽的同龄人身上扫过，漠鱼气炸刃千，矩干土漂贵这紧胜获，嘘，堪脯谁荡，，滴官逛卧怂命颊
少年缓缓抬起头来，露出一张有些清秀的稚嫩脸庞，漆黑的眸子木然的在周围那些嘲讽的同龄人身上扫过，领停猜头的疾，肃踢的鞭咳拒窝，先，。文快叫肃周，，袖鸟受胎具奕，
少年缓缓抬起头来，露出一张有些清秀的稚嫩脸庞，漆黑的眸子木然的在周围那些嘲讽的同龄人身上扫过，带，焦真绕，每奢烟涵，，蒂悸枪许漓，妇嚎恭互，评，熟揣拂彩定藉观
少年缓缓抬起头来，露出一张有些清秀的稚嫩脸庞，漆黑的眸子木然的在周围那些嘲讽的同龄人身上扫过，域刹着，趾，，之言骗，刻史屋搓析捏伐，尼左的了愤，扶述，米承宗，
少年缓缓抬起头来，露出一张有些清秀的稚嫩脸庞，漆黑的眸子木然的在周围那些嘲讽的同龄人身上扫过，，，，，，越p比胁渍苍袍，描王，部吞际，为贼佣，歇万，猾祟，勉雳
少年缓缓抬起头来，露出一张有些清秀的稚嫩脸庞，漆黑的眸子木然的在周围那些嘲讽的同龄人身上扫过，聊龇，僻次秀搂，炎建歼，茂，，胎胜，衫，引够被末肤堪半皆，守，，
少年缓缓抬起头来，露出一张有些清秀的稚嫩脸庞，漆黑的眸子木然的在周围那些嘲讽的同龄人身上扫过，财球速，朗，也，愧命救齐，怪俨慢也竖罢登械着妇，的尚匍妇锊爽悠狮
少年缓缓抬起头来，露出一张有些清秀的稚嫩脸庞，漆黑的眸子木然的在周围那些嘲讽的同龄人身上扫过，剑芒，，伏颤编于耻拂草，，，陌眩慕鲁圆胳追，咦瞎包鄙偷忧状鸿述教
少年缓缓抬起头来，露出一张有些清秀的稚嫩脸庞，漆黑的眸子木然的在周围那些嘲讽的同龄人身上扫过，黄的支酡脱棍，躯懒喃嬉扩述，，步插晒颗烛顺，噪扛，畸义，钥光干扩
少年缓缓抬起头来，露出一张有些清秀的稚嫩脸庞，漆黑的眸子木然的在周围那些嘲讽的同龄人身上扫过，霎挤萤砍毕昏泛，语担师义薯恨圾，撑的配先横定挡，叼药孰瞎叔着迫，
Epoch: 0, Batch: 0, Loss: 8.0796


KeyboardInterrupt: 

In [14]:
#ppl曲线的绘制
import matplotlib.pyplot as plt

# 检查是否有数据可以绘制
if len(ppl_train) > 0 and len(ppl_eval) > 0:
    # 创建一个新的图形窗口
    plt.figure(figsize=(10, 6))
    
    # 绘制训练集 PPL 曲线
    # 'b-' 表示蓝色实线，label 用于在图例中显示名称
    plt.plot(range(1, len(ppl_train) + 1), ppl_train, 'b-', label='Train PPL')
    
    # 绘制验证集 PPL 曲线
    # 'r-' 表示红色实线
    plt.plot(range(1, len(ppl_eval) + 1), ppl_eval, 'r-', label='Eval PPL')
    
    # 添加图表标题和轴标签
    plt.title('Perplexity (PPL) Curve over Epochs')
    plt.xlabel('Epoch')
    plt.ylabel('Perplexity (PPL)')
    
    # 设置 X 轴刻度，使其显示每一个 Epoch
    plt.xticks(range(1, len(ppl_train) + 1))
    
    # 添加网格线，方便观察数值变化
    plt.grid(True, linestyle='--', alpha=0.5)
    
    # 显示图例
    plt.legend()
    
    # 自动调整布局并显示图像
    plt.tight_layout()
    plt.show()
else:
    print("No PPL data to plot. Please run the training loop first.")

No PPL data to plot. Please run the training loop first.
